building a gpt :)

In [2]:
# we are using the Tiny Shakespeare dataset, downloadable here: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('tinyss.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# get character list
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_list = ''.join(chars)
print(f'numbber of unique characters: {vocab_size}, characters: {char_list}')

numbber of unique characters: 65, characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
hi there !


In [28]:
# create a mapping from characters to integers (character level tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("""hii there
jason""")) # spaces have index 1 while \n has index 0 
print(decode(encode("hii there"))) # note that even whitespace is tokenized

[46, 47, 47, 1, 58, 46, 43, 56, 43, 0, 48, 39, 57, 53, 52]
hii there


In [ ]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)

In [24]:
data.shape

torch.Size([1115394])

In [25]:
data[:1000]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [ ]:
# train/test split
n = int(0.9*len(data))
train_data = data[:n] # 90%
val_data = data[n:] # 10%

In [30]:
# setup context window 
block_size = 8
train_data[1:block_size+1]

tensor([47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
# with a larger context window, the model takes into account all tokens (in context window) to predict the next token. After that, the window "slides" right and cycle repeats
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


torch.int64

In [ ]:
torch.manual_seed(1337) # deterministic
batch_size = 4 # how many batches we train in parallel
block_size = 8 # size of context window

# training on the entire dataset would take too much time, train in random batches (chunks of randomly sampled text)
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # subtracting block size ensures we don't go out of bounds 
    # note: randint takes upper bound as default and lower bound as optional
    # note 2: size must be a tuple of ints (hence batch_size,)
    x = torch.stack([data[i:i+block_size] for i in ix]) # stack literally stacks the tensors (default axis = 0, vertically)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # +1 as targets are always the next token 
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb) # these are what we input into the transformer
print('targets:')
print(yb.shape)
print(yb) # these are what we expect as output from the transformer


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


## Testing with Bigram Model first


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C): for every token in input (batches x context window ("time") sized tensor), returns logits which represent a score for each possible next token

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape # unpacking dimensions of logits
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # auto reshapes into (B*T (predictions) x C (probability of all possible next tokens (scores) per prediction))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step as we want the most recent context
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution (generating the next token based off probability distribution)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [77]:
m = BigramLanguageModel(vocab_size) # create lookup table
logits, loss = m(xb, yb) # idx, targets
print(logits.shape) # for each of the 4x8 = 32 tokens, generate all 65 probabilities of the next token
print(loss)

torch.Size([256, 65])
tensor(4.6208, grad_fn=<NllLossBackward0>)


In [61]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens = 10)[0].tolist())) 
# creates the start token (\n) and asks the model to predict more tokens, returns a (1,11 tensor) that we use [0] on to flatten it into (11,)
# (11,) can then be converted to a python list, which ca then be decoded back into a string


gChzbQ?u!3


### we can definitely do better than this :)

In [ ]:
# create a PyTorch optimizer that handles gradient descent for us
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [99]:
batch_size = 32 # now the inputs and targets are 4 x 32 (print xb.shape)
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')
    
    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step() 

print(loss.item())

4.660644054412842


In [100]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


UbgRxMc:Iry br&NkJNOdhrBgqkEemwks,CCvVrGtVl $wRVL$llODP YZxz
bGoyo3xZrKgILKomi$X.-mocfm$YZ?ks&IAACQrgpBLgqsI
$YUhJyjhO&phNn:kLJ
;k cfAFLAFNeDvu3bn:uWq;;Bi$YSXoU.k
ZrM!fm$gE-tHF.KWdn:u
;ZBDeRbJNOGl
dy.

UzmwEEf?qSrnMR.lZk
MsmfjLn.-3s?Jw$3nMcf,,SVutaqC&VUdGFsFkW&bdZSzn.Cqk
x,wHupr&AsLriiMqSceTxXJAIGbWXfWEK3A,s;MU!p bm$:eU&rzLKnGhIDtE:Hd.fxYYrqCS
KMywQ3iPx'BjVo,!&W:Zkym
GFGcg;ivYUfKSX& NGgTRlW UNl!BY!TRt!j'VAtq;,,r&M!FsGh&n:IQuxu :OPr&V&Vl
UjnMZDl-rRxb.$onAFrwOLZseKTjphZzqpdm$V;Zr&E-reeh&x:QTj'Vy.R


## we can still do better c:

# Introducing self-attention

how self-attention works:

each token's embedding is multiplied with key, query and value matrices (which start out random and are learned during training) to get k,q,v vectors respectively 
key: what does the token have 
value: what does the token give
query: what is the token looking for

attention scores are calculated by taking the dot product of each token's query and each token's key, masked (to prevent tokens from interacting with future tokens), divided by the square root of head size (length of k,q,v vectors - larger size = more expressive, more computation)**, normalized (softmax) to get attention weights then multiplied by V to get the final output - a new representation of the token enriched with context from past tokens its powerful as this allows each token to communicate with past tokens in one step as compared to the old method of sequentially passing information (RNN), which is much faster and cost efficient.

**softmax is normalization: as vector length increases, dot product values get insanely large which may cause softmax saturation (gradients of softmax output become negligible)
softmax([1, 2, 3])    → [0.09, 0.24, 0.67]  # nice spread
softmax([10, 20, 30]) → [0.00, 0.00, 1.00]  # one token gets ALL attention, model stops learning
dividing by sqrt(head_size) keeps variance stable and keeps numbers from exploding while retaining the structure of how attention is spread through tokens.**
 

after the loss is calculated, the kqv matrices are then updated to get more accurate attention weights via the optimizer  